# 00 - Fit the Baseline Empirical Simulator

This notebook turns the combined Mendeley+NHANES KDM8 biological-age table into a small probabilistic simulator.

The active baseline is controlled by `MODEL_NAME` in `src/bioage_sbi/config.py`. It currently uses the common KDM8 lab-rich feature set so downstream notebooks can test whether richer observed data improves biological-age recovery.


## Setup

Keep reusable logic in `src/bioage_sbi`. The notebook should read like a walk-through, not a library file.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

SRC_CANDIDATES = [Path("../src"), Path("biological_age_sbi/experiment/src")]
for src in SRC_CANDIDATES:
    if (src / "bioage_sbi").exists():
        sys.path.insert(0, str(src.resolve()))
        break
else:
    raise FileNotFoundError("Could not find biological_age_sbi/experiment/src")

from bioage_sbi import (
    BASELINE_DATASET_NAME,
    FEATURE_SETS,
    MODEL_NAME,
    find_baseline_data_path,
    fit_empirical_model,
    load_baseline_data,
    prepare_model_frame,
    sample_component_model,
    save_empirical_model,
    split_train_holdout,
)
from bioage_sbi.simulator import make_component_functions

sns.set_theme(style="whitegrid")
np.set_printoptions(suppress=True)


## Load Baseline Data

`prepare_model_frame` keeps the active feature set, coerces modeling columns to numeric values, and drops rows with missing selected variables. The combined baseline keeps `source_dataset` in the raw frame so source composition can be checked before modeling.


In [ ]:
DATA_PATH = find_baseline_data_path()
raw_df = load_baseline_data(DATA_PATH)
model_frame = prepare_model_frame(raw_df)
source_counts = raw_df["source_dataset"].value_counts(dropna=False) if "source_dataset" in raw_df else pd.Series(dtype=int)

DATA_PATH, BASELINE_DATASET_NAME, raw_df.shape, model_frame.shape, source_counts


## Variable Set

This is the first diagnosis-expanded non-lab model. These variables are observable without drawing a new lab sample, but diagnosis variables such as diabetes may originally depend on medical testing.


In [ ]:
active_feature_set = FEATURE_SETS[MODEL_NAME]
active_continuous_columns = list(active_feature_set.continuous_columns)
active_binary_columns = list(active_feature_set.binary_columns)
active_model_columns = [*active_continuous_columns, *active_binary_columns]

variable_table = pd.DataFrame(
    [
        {
            "internal_name": "biological_age",
            "role": "target/prior",
            "active_model": True,
        },
        *[
            {
                "internal_name": col,
                "role": "continuous indicator",
                "active_model": True,
            }
            for col in active_continuous_columns
        ],
        *[
            {
                "internal_name": col,
                "role": "binary indicator/status",
                "active_model": True,
            }
            for col in active_binary_columns
        ],
    ]
)

active_feature_set.description, variable_table


## Train/Holdout Split

The empirical simulator is fitted on the training split. The holdout split is saved for the final real-world check in notebook `01`, so that check is not fully circular.


In [ ]:
train_frame, holdout_frame = split_train_holdout(model_frame, holdout_fraction=0.20, seed=2026)
train_frame.shape, holdout_frame.shape


## Descriptive Check

Before fitting a simulator, inspect ranges and prevalences. This tells us what the simulator should roughly reproduce.


In [ ]:
continuous_summary = train_frame[["biological_age", *active_continuous_columns]].describe(
    percentiles=[0.005, 0.01, 0.05, 0.5, 0.95, 0.99, 0.995]
).T
binary_prevalence = train_frame[active_binary_columns].mean().rename("prevalence").to_frame()

continuous_summary, binary_prevalence


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.histplot(train_frame["biological_age"], kde=True, ax=axes[0])
axes[0].set_title("Biological age prior data")

sns.regplot(
    data=train_frame.sample(min(len(train_frame), 2500), random_state=1),
    x="biological_age",
    y="bmi",
    lowess=True,
    scatter_kws={"alpha": 0.15, "s": 8},
    ax=axes[1],
)
axes[1].set_title("BMI versus biological age")

sns.regplot(
    data=train_frame.sample(min(len(train_frame), 2500), random_state=2),
    x="biological_age",
    y="sbp",
    lowess=True,
    scatter_kws={"alpha": 0.15, "s": 8},
    ax=axes[2],
)
axes[2].set_title("SBP versus biological age")

for ax in axes:
    sns.despine(ax=ax)
plt.tight_layout()


## Fit Empirical Probabilistic Model

The fitted simulator is still inspectable, but now uses one richer dependence layer:

- empirical quantile prior for biological age
- linear conditional model for BMI given biological age
- linear conditional model for SBP given biological age and BMI
- logistic conditional models for binary variables
- later binary variables can depend on earlier generated variables
- age-dependent latent risk factors induce shared residual structure across related indicators
- conditional Gaussian copula over PIT residuals captures remaining cross-indicator dependence


In [ ]:
empirical_model = fit_empirical_model(train_frame, model_name=MODEL_NAME)

dependence_model = empirical_model["dependence_model"]
copula_correlation = pd.DataFrame(
    dependence_model["correlation"],
    index=dependence_model["columns"],
    columns=dependence_model["columns"],
)

(
    empirical_model["model_name"],
    empirical_model["model_family"],
    empirical_model["fit_summary"],
    {
        "dependence_type": dependence_model["type"],
        "copula_enabled": dependence_model["enabled"],
        "min_eigenvalue_before_repair": dependence_model["min_eigenvalue_before_repair"],
    },
)


In [ ]:
continuous_models = pd.DataFrame(
    {
        name: {
            "features": ", ".join(spec["features"]),
            "residual_std": spec["residual_std"],
            "clip_low": spec["clip"][0],
            "clip_high": spec["clip"][1],
        }
        for name, spec in empirical_model["continuous_models"].items()
    }
).T

binary_models = pd.DataFrame(
    {
        name: {"features": ", ".join(spec["features"])}
        for name, spec in empirical_model["binary_models"].items()
    }
).T

continuous_models, binary_models, copula_correlation.round(2)


## Prior Predictive Check Of The Fitted Simulator

Sample directly from the reusable simulator components before using BayesFlow. The goal is not perfect reproduction, but obviously wrong ranges or prevalences should be caught here.


In [ ]:
simulated_frame = sample_component_model(empirical_model, num_samples=len(train_frame), seed=1234)
observed_key_by_column = empirical_model["observed_key_by_column"]
observed_keys = [observed_key_by_column[col] for col in empirical_model["columns"]]

simulated_observed = simulated_frame[["biological_age", *observed_keys]].rename(
    columns={observed_key_by_column[col]: col for col in empirical_model["columns"]}
)

simulated_observed.head()


In [ ]:
plot_columns = ["biological_age", *active_continuous_columns[:2]]
fig, axes = plt.subplots(1, len(plot_columns), figsize=(5 * len(plot_columns), 4))

for ax, col in zip(np.ravel(axes), plot_columns):
    sns.kdeplot(train_frame[col], label="Baseline train", ax=ax)
    sns.kdeplot(simulated_observed[col], label="Simulator", ax=ax)
    ax.set_title(col)
    ax.legend()
    sns.despine(ax=ax)

plt.tight_layout()


In [ ]:
prevalence_compare = pd.DataFrame(
    {
        "baseline_train": train_frame[active_binary_columns].mean(),
        "simulator": simulated_observed[active_binary_columns].mean(),
    }
)

ax = prevalence_compare.plot(kind="bar", figsize=(9, 4))
ax.set_ylim(0, 1)
ax.set_ylabel("Prevalence")
ax.set_title("Binary indicator prevalence")
sns.despine(ax=ax)
prevalence_compare


## Save Model And Split Data

Notebook `01` depends on these files. Rerun this notebook when the variable set or empirical model changes.


In [ ]:
PROCESSED_DIR = DATA_PATH.parent
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = PROCESSED_DIR / f"empirical_bioage_model_{MODEL_NAME}.json"
TRAIN_PATH = PROCESSED_DIR / f"baseline_train_{MODEL_NAME}.csv"
HOLDOUT_PATH = PROCESSED_DIR / f"baseline_holdout_{MODEL_NAME}.csv"

save_empirical_model(empirical_model, MODEL_PATH)
train_frame.to_csv(TRAIN_PATH, index=False)
holdout_frame.to_csv(HOLDOUT_PATH, index=False)

MODEL_PATH, TRAIN_PATH, HOLDOUT_PATH


## Component Functions Used In Notebook 01

This is the handoff: `01` will load the saved JSON and construct these same three simulator components.


In [ ]:
prior, bioindicator_model, observation_model = make_component_functions(empirical_model, seed=1234)
model_true_keys = [empirical_model["true_key_by_column"][col] for col in empirical_model["columns"]]
model_latent_keys = [f"latent_{name}" for name in empirical_model.get("latent_factors", {}).get("factors", {})]

example_prior = prior()
example_true = bioindicator_model(**example_prior)
example_observed = observation_model(**{key: example_true[key] for key in model_true_keys})
example_latents = {key: example_true[key] for key in model_latent_keys if key in example_true}

example_prior, example_latents, example_true, example_observed


## Model Boundary

The conditional Gaussian copula captures one global residual dependence structure. If this still fails to improve real-world recovery, the next suspect is not just dependence strength but misspecified conditional age signal: for example source-specific effects, age-bin-specific copulas, richer nonlinear marginals, or a mismatch between KDM-derived biological age and the simulator target.
